# 04 — Interpretability & Calibration
SHAP values, threshold optimisation, and business recommendations

In [1]:
import sys, os
# Move to project root so relative paths (data/, outputs/) resolve correctly
# Works whether the kernel starts from notebooks/ or the project root.
_here = os.path.abspath('.')
if os.path.basename(_here) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))
import warnings
warnings.filterwarnings('ignore')
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

from src.models.evaluator import find_optimal_threshold
from src.visualization.plots import plot_shap_summary, plot_calibration_curves

FIGURES = Path('outputs/figures')

## 1. Load Artifacts

In [2]:
split = joblib.load('outputs/reports/train_test_split.joblib')
X_test  = split['X_test']
y_test  = split['y_test']
feature_names = split['feature_names']

xgb_artifact = joblib.load('outputs/models/xgboost.joblib')
xgb_model    = xgb_artifact['model']
print('XGBoost metrics:', xgb_artifact['metrics'])

XGBoost metrics: {'roc_auc': 0.815066263659614, 'pr_auc': 0.6105955102617291, 'f1': 0.5605263157894737, 'f2': 0.5658873538788523, 'precision': 0.5518134715025906, 'recall': 0.56951871657754, 'accuracy': 0.7629524485450674}


## 2. SHAP Explainer

In [3]:
# Extract the raw XGBClassifier from the sklearn Pipeline
xgb_clf = xgb_model.named_steps['classifier']
X_test_scaled = xgb_model.named_steps['scaler'].transform(X_test)

explainer   = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_test_scaled)
print('SHAP values shape:', shap_values.shape)

SHAP values shape: (1409, 37)


## 3. Global Feature Importance — Beeswarm

In [4]:
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)
path = plot_shap_summary(shap_values, X_test_df, FIGURES)
img = mpimg.imread(path)
fig, ax = plt.subplots(figsize=(8, 7), dpi=100)
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Mean |SHAP| — Feature Ranking

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

mean_abs_shap = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_names,
).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(9, 6))
mean_abs_shap.plot(kind='barh', ax=ax, color=sns.color_palette('colorblind')[0])
ax.set_title('Top 15 Features by Mean |SHAP| Value', fontsize=13, fontweight='bold')
ax.set_xlabel('Mean |SHAP| value (impact on model output)')
plt.tight_layout()
plt.savefig('outputs/figures/shap_mean_abs.png', dpi=300, bbox_inches='tight')
plt.show()
print('Top 5 features:')
print(mean_abs_shap.tail(5).sort_values(ascending=False))

Top 5 features:
total_services                    1.030442
PaymentMethod_Electronic check    0.860089
tenure                            0.722746
MultipleLines_Yes                 0.664659
InternetService_Fiber optic       0.501755
dtype: float32


## 5. Calibration Curves

In [6]:
models_proba = {
    'XGBoost':            xgb_model.predict_proba(X_test)[:, 1],
}
# Add LR and RF if artifacts exist
for mname, fname in [('LogisticRegression', 'logistic_regression'), ('RandomForest', 'random_forest')]:
    art_path = Path(f'outputs/models/{fname}.joblib')
    if art_path.exists():
        art = joblib.load(art_path)
        models_proba[mname] = art['model'].predict_proba(X_test)[:, 1]

path = plot_calibration_curves(models_proba, y_test, FIGURES)
img = mpimg.imread(path)
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Optimal Decision Threshold (F2)

In [7]:
opt_threshold, best_f2 = find_optimal_threshold(xgb_model, X_test, y_test, metric='f2')
print(f'Optimal threshold: {opt_threshold:.2f}')
print(f'Best F2-score    : {best_f2:.4f}')

# Show F1/F2 vs threshold curve
import numpy as np
from sklearn.metrics import f1_score, fbeta_score

y_proba = xgb_model.predict_proba(X_test)[:, 1]
thresholds = np.arange(0.10, 0.91, 0.01)
f1s = [f1_score(y_test, (y_proba >= t).astype(int), zero_division=0) for t in thresholds]
f2s = [fbeta_score(y_test, (y_proba >= t).astype(int), beta=2, zero_division=0) for t in thresholds]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(thresholds, f1s, label='F1', lw=2)
ax.plot(thresholds, f2s, label='F2', lw=2)
ax.axvline(opt_threshold, color='red', linestyle='--', label=f'Optimal ({opt_threshold:.2f})')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Score')
ax.set_title('F1 and F2 vs Decision Threshold — XGBoost', fontweight='bold')
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig('outputs/figures/threshold_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

Optimal threshold: 0.10
Best F2-score    : 0.7331


## 7. Business Recommendations

Based on SHAP analysis of the XGBoost model trained on 7,043 IBM Telco customers:

### 1. Prioritise Month-to-Month Customers in the First 12 Months
Contract type and tenure are consistently among the top SHAP drivers. A month-to-month customer in their first year represents the highest churn risk — they face zero switching costs. **Action**: offer a discounted annual upgrade at month 3–6, before churn intent crystallises. Even converting 10% of this segment to one-year contracts reduces expected churn significantly.

### 2. Target Fiber Optic Customers Without Add-On Services
Fiber optic subscribers paying high monthly charges without Online Security, Tech Support or Device Protection show elevated SHAP values for churn. These customers perceive low value for money. **Action**: bundle one add-on service (e.g., Online Security) into the plan at no extra charge for the first 3 months. Increased perceived value reduces price-driven churn and cross-sells into stickier service tiers.

### 3. Use Calibrated Probabilities for Risk-Tiered Retention Campaigns
The calibration curve shows XGBoost's probabilities are reasonably reliable (points close to the diagonal). At the optimal F2 threshold, the model maximises recall — catching the most churners even at the cost of some false alarms. **Action**: segment customers into three risk tiers (Low: p < 0.3, Medium: 0.3–0.6, High: p > 0.6) and apply proportionally scaled interventions — automated email for medium risk, direct outreach + discount offer for high risk. This prevents wasting retention budget on customers who would have stayed anyway.